# 03 · Player Statistics — FBref

Wikipedia's player-level data was incomplete/unreliable, so player stats come from **FBref**:
minutes, goals, assists, position, nationality, age, cards, and per-90 metrics.

FBref is behind Cloudflare, so we use **undetected-chromedriver** (a real Chrome browser
with anti-bot patches) instead of plain `requests`.

> Output table in the model: **`player_Status`** (one row per player per season).


## 1. Prerequisites

```bash
pip install undetected-chromedriver selenium pandas beautifulsoup4 lxml
```
- Google Chrome installed.
- A Chrome window will open while scraping — leave it open.


## 2. Setup


In [ ]:
import undetected_chromedriver as uc
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO
import time, os, re

EXT_DIR = os.path.join('data', 'external')
os.makedirs(EXT_DIR, exist_ok=True)


## 3. Season → FBref URL

FBref standard stats page per season, e.g.
`https://fbref.com/en/comps/9/2023-2024/stats/2023-2024-Premier-League-Stats`


In [ ]:
def fbref_url(start_year):
    s = f'{start_year}-{start_year+1}'
    return f'https://fbref.com/en/comps/9/{s}/stats/{s}-Premier-League-Stats'

print(fbref_url(2023))


## 4. Launch Chrome (run once)


In [ ]:
options = uc.ChromeOptions()
options.add_argument('--disable-blink-features=AutomationControlled')
driver = uc.Chrome(options=options, version_main=None)
driver.set_page_load_timeout(60)
print('✅ Chrome launched — leave the window open.')


## 5. Helper — scrape the standard player stats table

FBref hides some tables inside HTML comments, so we strip comment markers before parsing.


In [ ]:
def scrape_player_stats(start_year):
    url = fbref_url(start_year)
    driver.get(url)
    WebDriverWait(driver, 30).until(EC.presence_of_element_located((By.TAG_NAME, 'table')))
    time.sleep(2)
    html = driver.page_source
    html = html.replace('<!--', '').replace('-->', '')  # reveal commented tables
    soup = BeautifulSoup(html, 'lxml')
    table = soup.select_one('table#stats_standard')
    if table is None:
        print(f'  ⚠️ stats table not found for {start_year}')
        return None
    df = pd.read_html(StringIO(str(table)))[0]
    # FBref uses a 2-level header; flatten it
    df.columns = [c[1] if isinstance(c, tuple) else c for c in df.columns]
    df = df[df['Rk'].apply(lambda x: str(x).isdigit())]  # drop repeated header rows
    df['SeasonKey'] = f'{start_year}\u2013{str(start_year+1)[-2:]}'
    return df


## 6. Scrape all seasons and combine


In [ ]:
frames = []
for y in range(2010, 2025):
    print(f'FBref {y}-{y+1} ...')
    df = scrape_player_stats(y)
    if df is not None:
        frames.append(df)
        print(f'  ✅ {len(df)} players')
    time.sleep(4)  # be polite / avoid rate limits

player_status = pd.concat(frames, ignore_index=True)
print('Total rows:', len(player_status))


## 7. Tidy columns and save

Keep the analytical columns and add the bridge keys used by the model
(`PlayerKey`, `ClubKey`, `SeasonKey`).


In [ ]:
keep = ['Player', 'Nation', 'Pos', 'Squad', 'Age', 'Min', 'Gls', 'Ast', 'CrdY', 'CrdR', 'SeasonKey']
ps = player_status[[c for c in keep if c in player_status.columns]].copy()
ps = ps.rename(columns={'Squad': 'Team', 'Gls': 'Goals', 'Ast': 'Assists'})

# bridge keys (apply the same clean_club / normalize used in notebook 02)
ps['PlayerKey'] = ps['Player']
ps['ClubKey']   = ps['Team']

ps.to_csv(os.path.join(EXT_DIR, 'fbref_player_status.csv'), index=False)
print('✅ Saved data/external/fbref_player_status.csv')


## 8. Close the browser


In [ ]:
driver.quit()
print('Done.')


## Notes

- FBref is the **reliable** source for player stats in this project; Wikipedia player tables were dropped.
- Per-90 metrics can be derived (e.g. `Goals / (Min/90)`) or pulled from FBref's per-90 columns.
- Respect FBref's rate limits — keep the delay between requests.
